# 📗 검색 전략과 벡터 저장소 종합 선정: 질문 유형 분석부터 결정 기록(ADR)까지

교안_01에서는 임베딩 모델 후보를 같은 평가셋으로 측정해 비교했습니다. 이번 교안에서는 남은 두 가지인 **검색 전략**과 **벡터 저장소**를 정하고, 지금까지 측정한 값을 모아 RAG 아키텍처 하나를 최종 선정합니다.

**이번 교안의 흐름**
1. **검색 전략(3절)**: 검색으로 답하는 질문(묘사·이름·조건)마다 세 검색 전략(`bm25`·`dense`·`hybrid`)의 Recall@5를 측정해 전략을 고릅니다. 관계·집계 질문은 검색이 아니라 구조화 질의로 답하므로 측정하지 않습니다(1절).
2. **벡터 저장소(4절)**: 서비스가 보낼 질의(벡터 검색, 키워드 검색, 메타데이터 필터, 관계 탐색, 개수 세기)를 저장소가 처리할 수 있는지로 고릅니다. 벡터를 float32·float16·binary 가운데 어떤 형식으로 저장할지도 정확도와 저장 공간을 측정해 정합니다.
3. **비용(5절)**: 비용을 색인 1회, 월 질의, 재색인으로 나눠 계산합니다.
4. **종합 선정(6절)**: 정확도·지연·비용·저장 공간 네 축을 후보표 하나에 모아 검색 전략 하나를 고릅니다. 임베딩 모델은 교안_01의 ADR-001에서 고른 KURE-v1을 그대로 씁니다.
5. **결정 기록**: 교안_01의 ADR-001에 이어 검색 전략(ADR-002), 벡터 저장소(ADR-003), 최종 아키텍처(ADR-004)를 ADR로 남깁니다.

<img src="images/00_lesson02_goals.png" width="1050" alt="검색 전략과 벡터 저장소를 고르고 RAG 아키텍처 선정하기: 1 질문 유형으로 전략, 전략으로 저장소 고르기, 2 제약으로 거르고 지배당한 후보를 지운 뒤 가중 점수로 고르기, 3 결정 문서(ADR)에 결정과 다시 검토할 조건 남기기">

**학습 목표**

1. 질문을 답에 필요한 근거에 따라 다섯 유형(묘사·이름·조건·관계·집계)으로 나누고, 검색으로 답하는 유형과 구조화 질의가 필요한 유형을 구분합니다.
2. 검색 전략의 정확도를 유형마다 측정해 고르고, 서비스가 보낼 질의를 처리하는 벡터 저장소와 벡터 저장 형식을 고릅니다.
3. 제약으로 후보를 거르고 지배당한 후보를 지운 뒤, 가중 점수와 민감도 분석으로 검색 전략을 고릅니다.
4. 결정과 근거, 결정을 다시 검토할 조건을 ADR로 기록합니다.

## 1. 검색 전략은 질문 유형으로 정한다

검색 전략을 비교하려면 먼저 질문을 나눌 기준이 있어야 합니다. 이 절에서는 질문 유형 다섯 가지를 정의하고, 그 가운데 검색 전략으로 답하는 유형과 다른 처리가 필요한 유형을 나눕니다.

### 1-1. 질문 유형 다섯 가지

같은 도메인 안에서도 질문마다 답을 찾는 데 필요한 근거가 다릅니다. 그래서 "법률이면 그래프"처럼 도메인 이름으로 정하지 않고 **질문 유형**으로 처리 방식을 정합니다.

<img src="images/06_question_shapes.png" width="1050" alt="사내 도서관 질문을 예로 든 다섯 칸 그림. 묘사(마음이 따뜻해지는 성장 소설)는 의미 검색, 이름(코스모스 책 있나요)은 어휘 검색 + 의미 검색, 조건(2020년 이후 나온 파이썬 전자책)은 필터 + 의미 검색, 관계(이 책 저자의 다른 책)와 집계(파이썬 책은 모두 몇 권인가요)는 구조화 질의로 이어진다">

여행지 안내 검색의 평가 질문 75개(유형마다 15개)는 다음과 같이 나뉩니다.

| 유형 | 예시 질문 | 답에 필요한 처리 | 이 교안에서 측정 |
|---|---|---|---|
| 묘사 | 희귀한 야생화와 곤충을 관찰하고 높은 흔들다리를 건너는 산속 명소는 어디야? | 뜻이 비슷한 문서 검색 | 측정 |
| 이름 | 창의문과 옛 성벽을 함께 볼 수 있는 산은 어디야? | 고유 이름이 들어간 문서 검색 | 측정 |
| 조건 | 강원도에서 소나무 숲과 백사장을 함께 즐길 수 있는 해변은? | 검색 + 지역 같은 메타데이터 필터 | 측정(필터 없이) |
| 관계 | 소돌아들바위공원 근처(10km 안)에서 함께 둘러볼 만한 해수욕장은 어디야? | 기준 관광지를 찾아 관계를 따라가는 구조화 질의 | 측정 안 함 |
| 집계 | 제주 관광지 가운데 자연관광으로 분류된 곳은 몇 곳이야? | 조건에 맞는 행을 세는 구조화 질의 | 측정 안 함 |

**검색으로 답하는 유형과 구조화 질의가 필요한 유형**
- 묘사·이름·조건은 문서를 찾으면 답이 나오는 질문이라, 3절에서 세 검색 전략(`bm25`·`dense`·`hybrid`)의 정확도를 비교합니다.
- 조건 질문의 지역 필터는 질문에서 조건을 뽑아야 붙일 수 있습니다. 그 일은 Self-Query(질의 변환 단원)가 LLM으로 하므로, 이 교안은 필터 없이 세 전략만 비교하고 필터를 지원하는지는 4절 저장소 기준에서 봅니다.
- 관계·집계는 문서를 찾는 것만으로 답이 나오지 않습니다. 관계는 기준 관광지를 이름으로 찾아 관계를 따라가야 하고(Text2Cypher·GraphCypherQAChain, 또는 SQL JOIN), 집계는 조건에 맞는 행을 모두 세야 합니다(Text-to-SQL). 둘 다 LLM이 질문을 질의로 바꿔야 하므로 이 교안에서는 측정하지 않고, 그 질의를 받을 수 있는 저장소인지만 4절에서 봅니다.

**앞 단원의 질문 분류와 비교**
- GraphRAG 변형 비교 단원의 단일 사실 질문은 묘사·이름·조건, 멀티홉 질문은 관계에 해당합니다.
- 자료 전체를 종합해야 답할 수 있는 **전역** 질문("이 지역 관광지들의 공통된 특색은?")은 개수를 세면 답이 나오는 집계 질문과 다릅니다. 이 평가셋에는 전역 질문이 없어 측정하지 않습니다.
- RAPTOR 단원과 GraphRAG 변형 비교 단원은 색인 단계에서 요약을 미리 만들어 두고 검색했습니다. 전역 질문이 들어오는 서비스라면 이 방법을 이번 교안의 선정 절차에 후보로 추가하고, 요약을 만드는 색인 비용도 비용 축에 넣습니다.

### 1-2. 측정하는 검색 전략과 측정하지 않는 기법

3절에서 측정하는 전략은 모든 질문에 똑같이 적용하는 세 가지입니다.

| 전략 | 찾는 방법 |
|---|---|
| `bm25` | 질문 낱말이 문서에 자주 나오고 전체에서 드물수록 높은 점수(어휘 검색) |
| `dense` | 질문·문서 임베딩의 유사도(의미 검색) |
| `hybrid` | `bm25`와 `dense`의 순위를 RRF로 합침(하이브리드 검색 단원) |

**이번 교안에서 측정하지 않는 앞 단원 기법**

이 교안은 LLM을 호출하지 않고 검색 결과만 비교합니다. 아래 기법은 LLM이나 리랭커 호출이 추가로 필요해 오늘은 사용하지 않고(성능도 측정하지 않습니다), 어떤 상황에서 후보로 추가하는지만 정리합니다.

| 묶음 | 기법 | 추가 비용 | 후보로 추가할 때 |
|---|---|---|---|
| 검색 보강 | 질의 변환, RAG-Fusion, CRAG·Self-RAG, 컨텍스트 압축 | 질문마다 LLM 호출 | 고른 전략으로도 정확도가 모자란 유형이 있을 때 그 유형에만 |
| 재정렬 | 리랭킹 | 질문마다 리랭커 계산 | 정답이 후보 안에 있는데 5위 밖으로 밀릴 때 |
| 조건 추출 | Self-Query | 질문마다 LLM 호출 | 질문에 지역·분류 같은 조건이 자주 들어 있을 때 |
| 구조화 질의 | Text-to-SQL, Text2Cypher(GraphCypherQAChain) | 질문마다 LLM 호출 | 관계를 따라가거나 개수를 세는 질문이 있을 때 |
| 그래프 확장 | VectorCypherRetriever | 그래프 DB 운영 | 벡터로 찾은 문서에서 관계를 여러 단계 따라가야 할 때 |
| 그래프 색인(전역) | Microsoft GraphRAG | 색인 때 LLM으로 개체·관계 추출과 커뮤니티 요약, 전역 검색은 질문마다 LLM 호출 | 자료 전체를 모아 답하는 전역 질문이 많을 때 |
| 그래프 색인(멀티홉) | LightRAG, HippoRAG | 색인 때 LLM으로 개체·관계 추출, 질문마다 LLM 호출 | 관계 표 없이 문서만 있는데 관계 질문이 많을 때 |

어느 기법을 어느 단계에 붙이는지는 부록(RAG 단계별 기법 체계)에 정리했습니다.

## 2. 실습 준비

유형마다 전략을 측정하려면 평가셋과, 검색에 쓸 BM25 색인·벡터 저장소를 먼저 준비해야 합니다.

교안_01에서 고른 임베딩 모델 KURE-v1을 미리 내려받습니다. 수업 시작 전에 이 셀만 실행해 두면 뒤에서 모델을 처음 불러올 때 기다리지 않습니다.

In [ ]:
from huggingface_hub import snapshot_download

# KURE-v1(약 2.3GB) 미리 내려받기(쓰지 않는 형식 onnx, openvino, pytorch_model.bin은 제외)
snapshot_download("nlpai-lab/KURE-v1", ignore_patterns=["onnx/*", "openvino/*", "pytorch_model.bin"])
print("KURE-v1 준비 완료")

### 2-1. 설정

임베딩 모델은 교안_01의 ADR-001에서 고른 **KURE-v1**(1024차원) 하나만 씁니다. 모델은 다시 비교하지 않고, 이 모델 위에서 검색 전략과 저장소를 정합니다.
- 임베딩 단원에서 쓴 `HuggingFaceEmbeddings`로 불러와 LangChain 저장소에 그대로 넘깁니다.
- 이 교안은 외부 API를 부르지 않습니다. `.env`에서는 4절의 Supabase 접속 주소만 읽습니다.

In [ ]:
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

# 단원 폴더(material_dir) 아래 data 폴더 경로
material_dir = Path(".")
data_dir = material_dir / "data"

# 접속 정보 읽기: 단원 .env 먼저, 없는 공통 값은 저장소 최상위 .env에서(이미 읽은 값은 유지)
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

# 교안_01 ADR-001에서 고른 임베딩 모델(벡터 길이를 1로 맞춰 거리 순서가 코사인 유사도 순서와 같음)
embedding_model = HuggingFaceEmbeddings(model_name="nlpai-lab/KURE-v1", encode_kwargs={"normalize_embeddings": True})
print("임베딩 모델:", embedding_model.model_name)

### 2-2. 자료와 평가 함수

교안_01과 같은 자료를 씁니다.
- 관광지 388건과 평가 질문 75개를 읽습니다.
- 측정에는 검색으로 답하는 묘사·이름·조건 질문 45개(`search_questions`)만 씁니다. 관계·집계 질문 30개는 구조화 질의 몫이라(1-1) 이 교안의 어느 측정에도 쓰지 않습니다.

In [ ]:
import json
import pandas as pd


# 관광지(JSON Lines라 한 줄씩 읽기)와 평가 질문
spots = [json.loads(line) for line in (data_dir / "본실습" / "spots.jsonl").read_text(encoding="utf-8").splitlines()]
questions = json.loads((data_dir / "본실습" / "questions.json").read_text(encoding="utf-8"))

# 측정에 쓸 질문: 검색으로 답하는 유형만(관계·집계는 쓰지 않음)
search_types = ["묘사", "이름", "조건"]
search_questions = [question for question in questions if question["question_type"] in search_types]

print("관광지:", len(spots), "| 질문:", len(questions), "| 측정에 쓰는 질문:", len(search_questions))

display(pd.DataFrame(questions)["question_type"].value_counts().rename("질문 수").to_frame())  # 유형별 질문 수

In [ ]:
# 검색 품질 평가 단원의 함수(predicted: 검색 결과 ID 순위순, relevant: 정답 ID 목록)
def recall_at_k(predicted, relevant, k):
    """전체 관련 문서 중 상위 k개가 찾아낸 비율(관련 수 / 전체 관련 수)."""
    hits = sum(1 for p in predicted[:k] if p in relevant)
    return hits / len(relevant)

### 2-3. BM25 색인과 벡터 저장소 준비

3절의 세 검색 전략은 모두 아래 두 검색기로 만듭니다. 둘 다 하이브리드 검색·Adaptive RAG 단원에서 쓴 LangChain 검색기입니다.
- **BM25**: `BM25Retriever`와 `kiwi_tokenize`(Adaptive RAG 단원과 같음)
- **의미 검색**: KURE-v1 벡터를 넣은 `Chroma` 저장소
- 관광지마다 `Document`를 만들고, metadata에 원문 ID(`source_id`)를 담아 검색 결과에서 바로 읽습니다.
- 색인에 걸린 시간을 측정해 둡니다(5절 비용 계산에 씀).

In [ ]:
import time
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma

kiwi = Kiwi()


def kiwi_tokenize(text):
    """문서와 질문에서 명사·외국어·숫자를 같은 기준으로 추출합니다."""
    # 반각 가운뎃점(･)을 일반 가운뎃점(·)으로 바꿔 같은 낱말이 다르게 쪼개지지 않게 함
    return [token.form.lower() for token in kiwi.tokenize(text.replace("･", "·"))
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]


# 관광지마다 Document 하나(본문은 제목 + 줄바꿈 + 개요, 교안_01과 같음)
documents = []
for spot in spots:
    documents.append(Document(id=spot["doc_id"], page_content=spot["title"] + "\n" + spot["text"], metadata={"source_id": spot["doc_id"]}))

index_seconds = {}  # 검색기별 색인 시간(초)

# BM25 색인(상위 10개 반환)
start = time.perf_counter()
bm25 = BM25Retriever.from_documents(documents, preprocess_func=kiwi_tokenize, k=10)
index_seconds["BM25"] = time.perf_counter() - start

# KURE-v1로 문서를 임베딩해 Chroma 메모리 컬렉션에 넣기(같은 이름의 컬렉션만 비우고 다시 넣음)
vector_store = Chroma(collection_name="travel_spots", embedding_function=embedding_model)
vector_store.reset_collection()
start = time.perf_counter()
vector_store.add_documents(documents)
index_seconds["KURE-v1"] = time.perf_counter() - start
print("문서:", len(documents), "| BM25 색인(초):", round(index_seconds["BM25"], 3), "| KURE-v1 임베딩과 적재(초):", round(index_seconds["KURE-v1"], 3))

## 3. 질문 유형별 검색 전략 측정

검색으로 답하는 세 유형(묘사·이름·조건)마다 세 검색 전략의 Recall@5를 측정하고, 최종 아키텍처에 넣을 검색 전략을 고릅니다.

### 3-1. 세 검색기 만들기

2-3의 BM25 색인과 Chroma 저장소로 세 검색기를 만듭니다. 모두 LLM을 호출하지 않습니다. `bm25`·`dense`는 상위 10개를, `hybrid`는 두 결과를 합친 순위(최대 20개)를 돌려줍니다.
- `dense`: `vector_store.as_retriever`로 만든 의미 검색기
- `hybrid`: `EnsembleRetriever`로 `bm25`와 `dense`의 순위를 RRF로 합친 검색기(가중치 0.5씩, 하이브리드 검색 단원과 같은 설정)

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

# 세 검색기(bm25·dense는 상위 10개, hybrid는 둘을 합친 순위), 이름으로 골라 쓰도록 dict에 담음
dense = vector_store.as_retriever(search_kwargs={"k": 10})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5], c=60, id_key="source_id")
retrievers = {"bm25": bm25, "dense": dense, "hybrid": hybrid}

### 3-2. 유형 × 전략 표

유형마다 어느 전략의 Recall@5가 가장 높은지 비교하려고, 묘사·이름·조건 질문 45개를 세 검색기로 모두 검색해 표로 모읍니다.
- 조건 질문도 필터 없이 검색합니다(1-1).
- 검색 한 번에 걸린 시간(`ms`)도 함께 기록해 6절 지연 축에 씁니다.

In [ ]:
strategies = ["bm25", "dense", "hybrid"]  # 단순한 전략부터 나열(점수가 같으면 앞의 전략을 고름)

dense.invoke(search_questions[0]["question"])  # 첫 검색의 준비 시간이 측정에 섞이지 않게 한 번 먼저 실행

# 질문마다 세 검색기로 검색해 (질문, 전략) 한 행씩: Recall@5와 검색 시간
rows = []
for question in search_questions:
    for strategy in strategies:
        start = time.perf_counter()
        found = retrievers[strategy].invoke(question["question"])
        elapsed_ms = (time.perf_counter() - start) * 1000
        found_ids = [doc.metadata["source_id"] for doc in found]
        rows.append({"question_id": question["question_id"], "question_type": question["question_type"], "strategy": strategy,
                     "recall_at_5": recall_at_k(found_ids, question["relevant_doc_ids"], 5), "ms": elapsed_ms})
results = pd.DataFrame(rows)

# 유형 x 전략 칸마다 평균 Recall@5
type_table = results.pivot_table(index="question_type", columns="strategy", values="recall_at_5").round(3)
type_table = type_table.reindex(index=search_types, columns=strategies)
display(type_table)

**결과 읽기** (각 칸은 유형별 평균 Recall@5)
- 이름 질문은 세 전략 모두 1.0입니다. 질문에 든 관광지 이름으로 키워드 검색도, 의미 검색도 정답 문서를 찾습니다.
- 묘사 질문은 `hybrid`가 0.533으로 `dense`(0.400)와 `bm25`(0.333)보다 높습니다. 두 순위를 합친 효과가 가장 크게 나는 유형입니다.
- 조건 질문은 세 전략이 0.816~0.853으로 차이가 0.04 안입니다. 필터 없이 질문 문장 그대로 검색한 값입니다.

### 3-3. 유형마다 다른 전략을 쓸 가치가 있는가

유형마다 가장 높은 전략을 골라 쓰면(유형별) 질문 유형을 고르는 라우터가 필요합니다. 라우터를 둘 만큼 정확도가 오르는지, 전략 하나로 모든 질문을 처리할 때와 비교합니다.
- 세 유형의 질문 수가 15개씩 같으므로 전체 정확도는 유형별 평균의 평균입니다.
- `유형별`은 평가셋의 유형 라벨로 전략을 고른 값이라, 라우터가 유형을 하나도 틀리지 않을 때의 상한입니다.

In [ ]:
routed_plan = type_table.idxmax(axis=1).to_dict()  # 유형마다 Recall@5가 가장 높은 전략(같으면 앞의 단순한 전략)
print("유형별 전략:", routed_plan)

# 전략 하나로 모든 질문을 처리할 때와 유형별로 고를 때의 전체 Recall@5
overall = type_table.mean().round(3).to_dict()
overall["유형별"] = round(float(type_table.max(axis=1).mean()), 3)
display(pd.Series(overall, name="전체 Recall@5").to_frame())

**결과 읽기**
- 각 행은 3-2 표의 세 칸 평균입니다. 전략 하나로 처리하면 `hybrid`가 0.794((0.533 + 1.000 + 0.849) ÷ 3)로 가장 높고, 유형별로 고른 상한은 행마다 가장 큰 값의 평균 0.795((0.533 + 1.000 + 0.853) ÷ 3)로 차이가 0.001입니다.
- 이 차이는 조건 질문에서 `bm25`(0.853)가 `hybrid`(0.849)보다 조금 높은 데서만 나옵니다.
- 라우터가 유형을 하나도 틀리지 않아도 얻는 것이 0.001이므로, 라우터를 만들고 운영할 이유가 없습니다. 검색 질문은 모두 `hybrid` 하나로 처리합니다.

#### ADR-002: 검색 전략으로 hybrid 하나 사용

- **상태**: 채택
- **맥락**: 여행지 안내 챗봇의 검색 질문(묘사·이름·조건)에 쓸 검색 전략을 정해야 한다. 임베딩 모델은 ADR-001의 KURE-v1이다. 후보는 BM25(`bm25`), KURE-v1 의미 검색(`dense`), 둘의 순위를 RRF로 합친 `hybrid`, 그리고 질문 유형마다 가장 높은 전략을 골라 쓰는 방식(유형을 판단하는 라우터 필요)이다. 관계·집계 질문은 구조화 질의(Text-to-SQL, Text2Cypher)가 맡으므로 이 결정에 넣지 않는다.
- **결정**: 검색 질문 전체의 검색 전략으로 `hybrid`(BM25와 KURE-v1 결과를 같은 가중치로 RRF 결합) 하나를 쓴다.
    - 근거: 측정값(검색 질문 45문항, 유형마다 15문항, 값은 Recall@5):

        | 후보 | 묘사 | 이름 | 조건 | 전체 | 판정 |
        |---|---|---|---|---|---|
        | `bm25` | 0.333 | 1.000 | 0.853 | 0.729 | 정확도 부족 |
        | `dense` | 0.400 | 1.000 | 0.816 | 0.739 | 정확도 부족 |
        | 유형별(라우터) | 0.533 | 1.000 | 0.853 | 0.795 | 라우터를 둘 만한 차이 없음 |
        | **`hybrid`** | **0.533** | **1.000** | **0.849** | **0.794** | **채택** |

        - `bm25`·`dense`: 전체 Recall@5가 `hybrid`보다 0.065·0.055 낮다. 차이는 주로 묘사 질문에서 나고, `dense`는 조건 질문에서도 0.033 낮다.
        - 유형별: 라우터가 유형을 하나도 틀리지 않을 때의 상한이 0.795로 `hybrid`보다 0.001 높을 뿐이다(조건 질문의 0.853 대 0.849).
        - `hybrid`: 묘사 질문에서 가장 높고(0.533), 이름·조건은 가장 높은 전략과 같거나 0.004 안이다.
- **결과**
    - 얻는 것: 라우터 없이 전략 하나(`hybrid`)로 세 유형을 처리한다. `dense` 하나만 쓸 때보다 묘사 질문 Recall@5가 오른다(0.400 -> 0.533).
    - 감수하는 것: 질문마다 두 검색기를 모두 실행하므로 키워드 색인과 벡터 인덱스를 함께 운영한다. 조건 질문은 `bm25` 하나보다 조금 낮다(0.853 대 0.849).
    - 다시 검토할 조건:
        - 평가 질문이나 관광지 문서가 바뀜 -> 유형 × 전략 표를 다시 측정하고, 유형별 상한과 `hybrid`의 차이가 커졌으면 라우터를 다시 비교한다.
        - ADR-001의 임베딩 모델이 바뀜 -> 유형 × 전략 표를 새 모델로 다시 측정한다.
        - Self-Query 필터를 실제로 붙여 운영함 -> 조건 질문을 필터를 붙인 상태로 세 전략을 다시 측정하고 유형별 상한과 다시 비교한다.

## 4. 벡터 저장소 선택: 전략에 필요한 질의를 처리하는가

3절에서 고른 전략을 실행할 저장소를 고릅니다. 기준은 "어느 제품이 빠른가"가 아니라 "고른 전략과 서비스 질문에 필요한 질의(벡터 검색, 키워드 검색, 메타데이터 필터, 관계 탐색, 개수 세기)를 처리할 수 있는가"입니다.

**여행지 안내 챗봇의 가정** (앞 네 줄은 교안_01과 같고, 뒤 두 줄은 이 교안에서 더함)
- 관광지 개요를 끝까지 읽고 답해야 합니다.
- 이용자 질문을 회사 밖 API로 보내지 않습니다.
- '부산의 박물관'처럼 지역으로 거르는 질문과 '근처'를 묻는 질문이 많습니다.
- 회사는 이미 PostgreSQL을 운영합니다.
- 서비스 규모는 관광지 100만 건, 월 질문 1만 건으로 봅니다(5·6절 계산에 씀).
- 검색 단계 지연은 300ms, 검색에 쓰는 메모리(벡터·인덱스와 BM25 색인)는 64GB를 넘으면 안 됩니다(4·6절 제약).

첫째·둘째 가정은 교안_01에서 KURE-v1을 고른 근거이고(ADR-001), 저장소는 셋째·넷째 가정으로 고릅니다.

**벡터 저장소 세 종류** (모두 앞 단원에서 사용했습니다)
- **전용 벡터 DB**: 벡터 검색만을 위한 DB입니다. Chroma·Qdrant가 여기에 속합니다.
- **pgvector**: PostgreSQL 테이블의 벡터 열이라 다른 테이블과 함께 SQL 한 문장으로 찾습니다.
- **Neo4j Vector**: 그래프 노드에 만든 벡터 인덱스로 찾고, 노드 사이 관계를 따라 결과를 넓힙니다.

<img src="images/04_three_stores.png" width="1050" alt="도서관 예시로 본 벡터 저장소 세 종류: 전용 벡터 DB는 '이 책과 비슷한 책'처럼 대량 유사도 검색, pgvector는 '지금 빌릴 수 있는 비슷한 책'처럼 책 테이블과 대출 테이블을 JOIN, Neo4j Vector는 '이 책 저자의 다른 책'처럼 벡터로 찾은 책에서 관계를 따라 확장">

### 4-1. 서비스가 보낼 질의

3절의 검색 전략과 1-1에서 따로 처리한다고 한 조건 필터·관계·집계를 저장소가 처리할 질의로 바꾸면 아래 표와 같습니다. 각 칸은 그 저장소 하나로 처리하는 방법이며, 모두 앞 단원에서 사용한 기능입니다.
- `한 질의`: 저장소의 질의 한 번으로 답이 나옵니다.
- `코드로 우회`: 저장소 기능은 있지만 밖에서 만든 값(형태소 분석 결과 등)을 넣거나, 꺼낸 결과를 파이썬 코드로 이어 붙여야 합니다.
- `불가`: 저장소에 그 기능이 없어 다른 도구가 대신해야 합니다.
- 관계·집계 질문은 검색이 아니라 Text-to-SQL·Text2Cypher가 만든 질의로 답하므로, 여기서는 저장소가 그 질의를 받을 수 있는지만 봅니다.

| 필요한 곳 | 필요한 질의 | 전용 벡터 DB(Chroma) | pgvector | Neo4j Vector |
|---|---|---|---|---|
| 의미 검색(`dense`, `hybrid`의 벡터 쪽) | 질문과 가까운 벡터 k개 | 한 질의(`query`) | 한 질의(`ORDER BY` 거리) | 한 질의(`SEARCH`) |
| 키워드 검색(`bm25`, `hybrid`의 키워드 쪽) | 점수 순위를 매기는 키워드 검색 | 불가(낱말 포함 여부만 거름, 순위는 밖의 BM25가 대신) | 코드로 우회(전문 검색 + 한국어 형태소 분석) | 한 질의(전문 인덱스) |
| 조건 질문(Self-Query가 뽑은 필터) | 지역 조건 안에서 가까운 벡터 k개 | 한 질의(`where`) | 한 질의(`WHERE`, 4-1 아래 주의 참고) | 한 질의(필터용 인덱스의 `SEARCH ... WHERE`) |
| 관계 질문(근처 관광지) | 기준 관광지의 근처 관광지를 찾아 분류로 거르기 | 코드로 우회(관계를 파이썬에서 따로 읽음) | 한 질의(근처 관계 테이블 JOIN) | 한 질의(`MATCH (a)-[:NEAR]-(b)`) |
| 집계 질문 | 조건에 맞는 행 세기 | 코드로 우회(`get`으로 꺼내 셈) | 한 질의(`count(*)`) | 한 질의(`count(n)`) |

**키워드 검색(BM25)은 어디서 실행하나?**
- 3절의 `BM25Retriever`는 문서 전체를 파이썬 메모리에 올려 계산하므로, 운영에서는 DB의 전문 검색이나 별도 검색 엔진(OpenSearch 등)을 씁니다.
- [Neo4j](https://neo4j.com/docs/cypher-manual/current/indexes/semantic-indexes/full-text-indexes/)는 Lucene 전문 인덱스를 제공합니다.
- [PostgreSQL](https://www.postgresql.org/docs/current/textsearch-intro.html) 전문 검색은 한국어 형태소 분석을 따로 해야 합니다.
- 어느 쪽이든 한국어 키워드 검색 품질은 분석기(형태소 분석) 설정에 달려 있습니다.
- 로컬 Chroma의 `where_document`(`$contains`)는 낱말이 들어 있는 문서만 거르고 점수 순위는 매기지 않습니다([문서](https://docs.trychroma.com/docs/querying-collections/full-text-search)). 그래서 BM25 같은 다른 도구가 순위를 대신 매겨야 해 `불가`입니다. 순위를 매기는 검색 API(희소 벡터 BM25)는 Chroma Cloud에만 있습니다.
- pgvector는 PostgreSQL 전문 검색으로 순위(`ts_rank`)까지 매기지만, 한국어는 형태소 분석을 밖에서 해 넣어야 해 `코드로 우회`입니다.

**표 읽기**
- pgvector는 키워드 검색을 뺀 모든 줄을 SQL 한 질의로 처리합니다.
- Neo4j Vector는 모든 줄을 한 질의로 처리하지만, 회사가 운영하지 않는 그래프 DB를 새로 들여야 합니다.
- Chroma는 관계 탐색·개수 세기를 파이썬 코드로 따로 구현해야 하고, 키워드 순위 검색은 다른 도구에 맡겨야 합니다.
- 조건 질문의 필터는 pgvector에서 주의할 점이 하나 있습니다. HNSW 인덱스를 쓰면 `WHERE` 조건을 인덱스가 꺼낸 후보에 나중에 적용(사후 필터)하므로, 조건에 맞는 문서가 적으면 요청한 개수보다 적은 결과를 오류 없이 돌려줍니다. 이때는 `hnsw.iterative_scan`을 켜서 결과가 찰 때까지 인덱스를 더 훑게 합니다([pgvector Filtering](https://github.com/pgvector/pgvector#filtering)).

### 4-2. 규모에 따른 저장소 전환 기준

388건은 어느 저장소에도 문제없이 들어가므로, 문서가 늘어날 때 어느 규모에서 저장소를 다시 검토할지 정해 둡니다. 또 6절에서 100만 건의 저장 공간을 추정하려면 HNSW 인덱스가 벡터에 더하는 크기를 알아야 하므로, 관광지 벡터를 Supabase의 pgvector 테이블에 넣고 실제 바이트 수를 측정합니다.

**접속**: **psycopg2**(파이썬에서 PostgreSQL에 SQL을 보내는 드라이버)와 `SUPABASE_DB_URL`로 접속합니다(주소 만드는 법은 이 폴더의 `Supabase_접속_주소_만들기.md`). 앞 단원에서 쓴 supabase-py는 REST API라 `CREATE EXTENSION`·`CREATE INDEX` 같은 SQL을 보낼 수 없기 때문입니다. 연결 하나를 4절 끝까지 씁니다.

In [ ]:
import os
from urllib.parse import urlsplit
import psycopg2

connection = psycopg2.connect(os.environ["SUPABASE_DB_URL"])  # .env의 Supabase 접속 주소로 연결 하나 열기(주소에 비밀번호가 있어 출력하지 않음)

connection.autocommit = True  # 문장마다 바로 반영해 CREATE, INSERT 뒤에 commit을 따로 부르지 않음


def run_sql(query, params=None):
    """값을 %s 매개변수로 전달하고 SQL 결과를 딕셔너리 리스트로 반환합니다."""
    with connection.cursor() as cursor:
        cursor.execute(query, params)

        # CREATE, INSERT처럼 돌려줄 행이 없는 문장
        if cursor.description is None:
            return []

        # 열 이름을 키로 한 딕셔너리 목록
        columns = [column.name for column in cursor.description]
        return [dict(zip(columns, row)) for row in cursor.fetchall()]


def vector_literal(vector):
    """숫자 목록을 pgvector가 읽는 '[0.1,0.2,...]' 문자열로 바꿔 반환합니다."""
    return "[" + ",".join([str(value) for value in vector]) + "]"  # psycopg2가 목록을 vector로 바로 못 바꿔 글자로 넘기고 SQL에서 ::vector로 바꿈


# 호스트와 포트만 출력
supabase_address = urlsplit(os.environ["SUPABASE_DB_URL"])
print("Supabase 연결 완료. 호스트:", supabase_address.hostname, "| 포트:", supabase_address.port)

관광지 벡터 테이블 `spot_vectors`를 새로 만들고 KURE-v1 문서 벡터를 넣습니다. 질문 벡터는 4-3 정밀도 측정에서 씁니다.
- 문서 벡터는 다시 임베딩하지 않고 Chroma에서 `get(include=["embeddings"])`로 꺼냅니다. 측정에 쓰는 질문 45개만 `embed_query`로 임베딩합니다.
- 확장은 `WITH SCHEMA extensions`로 켭니다. 빼면 대시보드에서 켤 때와 다른 public 스키마에 설치됩니다.
- 원격 DB라 행을 하나씩 보내면 왕복이 388번 생기므로 `execute_values`로 100행씩 묶어 보냅니다.

In [ ]:
from psycopg2.extras import execute_values

run_sql("CREATE EXTENSION IF NOT EXISTS vector WITH SCHEMA extensions")  # vector 확장을 extensions 스키마에 켜기(이미 켜져 있으면 그대로)

# Chroma에 넣어 둔 KURE-v1 문서 벡터를 원문 ID로 찾을 수 있게 꺼내기, 질문은 새로 임베딩
stored = vector_store.get(include=["embeddings"])
doc_vectors = dict(zip(stored["ids"], stored["embeddings"]))
question_vectors = [embedding_model.embed_query(question["question"]) for question in search_questions]

# 원문 ID와 벡터만 담은 spot_vectors 테이블을 새로 만들기
run_sql("DROP TABLE IF EXISTS spot_vectors")
run_sql("CREATE TABLE spot_vectors (doc_id text PRIMARY KEY, embedding vector(1024))")

# 관광지마다 (원문 ID, 벡터 글자) 행 만들기(벡터는 INSERT 안에서 ::vector로 바꿈)
spot_rows = [(spot["doc_id"], vector_literal(doc_vectors[spot["doc_id"]])) for spot in spots]

# 100행씩 INSERT 한 문장에 묶어 보내기
with connection.cursor() as cursor:
    execute_values(cursor, "INSERT INTO spot_vectors VALUES %s", spot_rows, template="(%s, %s::vector)", page_size=100)

# 코사인 거리용 HNSW 인덱스 만들기
run_sql("CREATE INDEX spot_vectors_embedding_hnsw ON spot_vectors USING hnsw (embedding vector_cosine_ops)")
print("pgvector 적재:", run_sql("SELECT count(*) AS n FROM spot_vectors")[0]["n"], "건")

적재한 `spot_vectors` 테이블에서 벡터 하나와 HNSW 인덱스의 크기를 측정합니다.

- `pg_column_size`는 값 하나, `pg_relation_size`는 테이블이나 인덱스 하나의 바이트입니다.
- 관광지 하나를 청크로 나누지 않고 통째로 임베딩해 문서 1개 = 청크 1개라, 벡터 수는 문서 수(388)와 같습니다.
- pgvector 문서는 `vector` 값 하나를 4×차원+8바이트로 안내합니다.

In [ ]:
# 벡터 하나의 평균 바이트와 HNSW 인덱스 크기를 한 번에 구하기
size = run_sql("""
SELECT
    (SELECT avg(pg_column_size(embedding)) FROM spot_vectors) AS vector_value,
    pg_relation_size('spot_vectors_embedding_hnsw') AS vector_index""")[0]

# 원시 벡터는 교안_01처럼 벡터 수(문서 388건) x 1024차원 x 4바이트(KB)
raw_kb = len(spots) * 1024 * 4 // 1024
index_kb = size["vector_index"] // 1024

# 원시 벡터 대비 (원시 벡터 + 인덱스) 배수, 6절에서 100만 건 저장 공간을 추정할 때 씀
vector_index_factor = round((raw_kb + index_kb) / raw_kb, 1)
print("벡터 하나 실측:", float(size["vector_value"]), "B | 공식(4 x 차원 + 8):", 4 * 1024 + 8, "B")
print("원시 벡터:", raw_kb, "KB | HNSW 인덱스:", index_kb, "KB | 원시 벡터 + 인덱스:", vector_index_factor, "배")

**결과 읽기**
- 벡터 하나 실측값(4100바이트)이 공식(4104바이트)과 거의 같습니다.
- HNSW 인덱스가 더해져 원시 벡터의 3.0배입니다. 6절은 이 배수(`vector_index_factor`)로 100만 건의 저장 공간을 추정합니다.

**규모별 전환 기준** ("벡터만" 크기는 교안_01 5-1의 식으로 계산한 KURE-v1 1024차원 값, 규모는 벡터 수)

| 규모 | 판단할 점 | 근거 |
|---|---|---|
| 10만 건 | 벡터만 약 0.4GB. 이미 운영 중인 DB에 붙이는 쪽부터 검토 | 교안_01 5-1 식 |
| 100만 건 | 벡터만 약 4.1GB. 인덱스까지 메모리에 올릴 수 있는지가 관건 | [AWS 가이드](https://docs.aws.amazon.com/prescriptive-guidance/latest/choosing-an-aws-vector-database-for-rag-use-cases/vector-db-comparison.html): pgvector는 수백만 건(장비에 따라). [Neo4j 매뉴얼](https://neo4j.com/docs/operations-manual/current/performance/vector-index-memory-configuration/): 768차원 100만 건 인덱스 메모리 3.1GB(양자화 없음), 0.16GB(binary) |
| 1억 건 | 벡터만 400GB 이상. 양자화와 여러 서버 분산이 기본인 전용 벡터 DB 검토 | [Qdrant 양자화](https://qdrant.tech/documentation/manage-data/quantization/): scalar로 메모리 4배, binary로 32배 절감 |

**차원 제약**: pgvector `vector`의 HNSW는 2,000차원까지라 KURE-v1(1024차원)은 그대로 들어갑니다. 이보다 큰 차원은 `halfvec`(4,000차원까지)으로 저장하거나 줄여야 합니다.

### 4-3. 저장 형식 정하기: 정밀도와 저장 공간

저장소 후보를 좁혔으면 벡터를 어떤 형식으로 저장할지도 정해야 합니다. 값 하나를 더 적은 비트로 저장하면 저장 공간이 줄지만, 값이 거칠어져 검색 순위가 바뀔 수 있습니다.

**정밀도 낮추기(양자화)란?**
- 벡터 값 하나를 더 적은 비트로 저장하는 방법입니다. pgvector는 열 형식으로 고릅니다.

| 형식 | 값 하나 | 1024차원 벡터 1개 | 저장하는 것 | pgvector 열 형식 |
|---|---|---|---|---|
| float32 | 4바이트 | 4,096바이트 | 원래 값 | `vector` |
| float16 | 2바이트 | 2,048바이트 | 자릿수를 줄여 반올림한 값 | `halfvec` |
| binary | 1비트 | 128바이트 | 부호만(0보다 크면 1, 아니면 0) | `bit` |

- `spot_vectors`의 float32 벡터를 세 형식으로 담은 `spot_quantized_vectors` 테이블을 만들고, 실제 크기와 의미 검색 정확도를 비교합니다.
- 질문도 같은 형식으로 바꿔 검색합니다. `<=>`는 코사인 거리, `<~>`는 해밍 거리(부호가 다른 칸 수)이고 둘 다 작을수록 가깝습니다.
- 이 테이블에는 인덱스를 만들지 않아 모든 행을 비교하는 정확 검색이 됩니다. 그래서 결과 차이는 저장 형식에서만 생깁니다.

In [ ]:
# 4-2의 float32 벡터를 세 형식 열로 복사한 테이블(인덱스 없음)
run_sql("DROP TABLE IF EXISTS spot_quantized_vectors")
run_sql("""
CREATE TABLE spot_quantized_vectors (
    doc_id text PRIMARY KEY,  -- 관광지 ID
    full_vector vector(1024), -- float32 원래 값
    half halfvec(1024),       -- float16으로 반올림한 값
    sign bit(1024)            -- 부호만 0과 1로 남긴 값
)""")
run_sql("""
INSERT INTO spot_quantized_vectors
SELECT doc_id, embedding, embedding::halfvec(1024), binary_quantize(embedding)::bit(1024)  -- 같은 벡터를 세 형식으로
FROM spot_vectors""")

# 형식마다 벡터 1개가 DB에서 차지하는 평균 바이트(pg_column_size: 값 하나의 저장 크기)
stored_bytes = run_sql("""
SELECT avg(pg_column_size(full_vector)) AS float32,
       avg(pg_column_size(half)) AS float16,
       avg(pg_column_size(sign)) AS binary
FROM spot_quantized_vectors""")[0]
print("벡터 1개(바이트):", {name: float(value) for name, value in stored_bytes.items()})

In [ ]:
import numpy as np

# 형식마다 질문과 가까운 5개를 찾는 SQL(질문 벡터도 같은 형식으로 바꿈)
format_sql = {
    "float32": "SELECT doc_id FROM spot_quantized_vectors ORDER BY full_vector <=> %s::vector LIMIT 5",
    "float16": "SELECT doc_id FROM spot_quantized_vectors ORDER BY half <=> %s::halfvec(1024) LIMIT 5",
    "binary": "SELECT doc_id FROM spot_quantized_vectors ORDER BY sign <~> binary_quantize(%s::vector)::bit(1024) LIMIT 5",
}

# 검색으로 답하는 질문 45개(묘사·이름·조건)를 의미 검색해 형식마다 Recall@5와 맞힌 문항(float32 대비 잃은·새로 맞힌 문항 포함)
precision_rows = []
for name, sql in format_sql.items():
    recalls = []
    hit_ids = set()
    for question, vector in zip(search_questions, question_vectors):
        found = [row["doc_id"] for row in run_sql(sql, (vector_literal(vector),))]
        recall = recall_at_k(found, question["relevant_doc_ids"], 5)
        recalls.append(recall)
        if recall > 0:
            hit_ids.add(question["question_id"])
    if name == "float32":
        float32_hits = hit_ids  # 첫 행 float32가 비교 기준
    precision_rows.append({"형식": name, "벡터 1개(바이트)": float(stored_bytes[name]),
                           "100만 건(GB)": round(1_000_000 * float(stored_bytes[name]) / 1e9, 3),
                           "Recall@5": round(float(np.mean(recalls)), 3), "맞힌 문항": len(hit_ids),
                           "잃은 문항": len(float32_hits - hit_ids), "새로 맞힌 문항": len(hit_ids - float32_hits)})
display(pd.DataFrame(precision_rows))

**결과 읽기**
- float16은 저장 공간이 절반(벡터 1개 2,052바이트)인데 맞힌 문항이 float32와 같습니다(35문항, 잃은 문항 0).
- binary는 30분의 1(136바이트)이지만 낮춘 값만으로 검색하면 2문항을 잃습니다(35 -> 33, Recall@5 0.739 -> 0.679).
- 그래서 메모리가 모자라면 이 평가셋에서 잃은 문항이 없었던 float16부터 쓰고, binary는 아래의 원본 + 사본 방식으로 씁니다.

**실무에서는 두 방식으로 씁니다**

| | 낮춘 값만 저장 | 원본 + 낮춘 사본(다시 점수 매기기) |
|---|---|---|
| 저장 | 낮춘 값만 | 원본은 디스크, 낮춘 사본은 메모리 |
| 검색 | 낮춘 값으로 바로 상위 10개 | 낮춘 사본으로 후보를 넉넉히(예: 30개) 뽑고, 그 후보만 원본으로 다시 점수를 매겨 상위 10개 |
| 주로 쓰는 정밀도 | float16(`halfvec`) | binary(`bit`) |

- 위 측정은 낮춘 값만으로 검색한 결과입니다. 손실이 큰 binary는 원본을 함께 두고 후보만 다시 점수를 매겨 손실을 줄입니다.
- 원본까지 두면 저장 총량은 오히려 늘지만, 검색 속도를 정하는 메모리에는 원본의 32분의 1인 binary만 올립니다.

**확인 퀴즈**: KURE-v1 벡터 100만 개를 binary로 저장하면 벡터 저장 공간은 얼마일까요?

<details><summary>정답 보기</summary>

벡터 하나가 1,024비트 = 128바이트이므로 128 × 100만 = 128,000,000바이트, 약 0.128GB입니다. float32(4.096GB)의 32분의 1입니다. 실제 저장값은 값마다 붙는 머리 정보(8바이트) 때문에 136바이트로, 약 30분의 1입니다.

</details>

### 4-4. 여행지 검색의 저장소와 저장 형식 결정

#### ADR-003: 벡터 저장소로 pgvector 사용(float32 저장)

- **상태**: 채택
- **맥락**: ADR-002의 `hybrid`(의미 검색 + 키워드 검색)와 서비스가 보낼 질의(지역 필터, 근처 관계, 개수 세기)를 처리할 벡터 저장소와 벡터 저장 형식을 정해야 한다. 회사는 이미 PostgreSQL을 운영하고, 문서 100만 건의 KURE-v1 벡터(1024차원)와 키워드 색인을 검색용 메모리 64GB 안에 올려야 한다. 저장소 후보는 전용 벡터 DB(Chroma), pgvector, Neo4j Vector이고, 저장 형식 후보는 float32·float16·binary다.
- **결정**: 벡터 저장소로 pgvector를 쓰고, 벡터는 `vector`(float32, 1024차원) 열에 HNSW 인덱스로 저장한다. 키워드 검색은 같은 PostgreSQL의 전문 검색(`ts_rank`, 한국어 형태소 분석은 앱에서 kiwi로)으로 실행한다.
    - 근거 1: 서비스가 보낼 질의(4-1)를 저장소 하나로 처리하는 방법:

        | 후보 | 의미 검색 | 키워드 검색 | 지역 필터 | 근처 관계 | 개수 세기 | 판정 |
        |---|---|---|---|---|---|---|
        | Chroma | 한 질의 | 불가 | 한 질의 | 코드로 우회 | 코드로 우회 | 우회가 많음 |
        | Neo4j Vector | 한 질의 | 한 질의 | 한 질의 | 한 질의 | 한 질의 | 새 DB 운영 필요 |
        | **pgvector** | **한 질의** | 코드로 우회 | **한 질의** | **한 질의** | **한 질의** | **채택** |

        - Chroma: 근처 관계·개수 세기를 파이썬 코드로 메워야 하고, 키워드 순위 검색은 다른 도구에 맡겨야 한다.
        - Neo4j Vector: 키워드 검색까지 다섯 질의를 한 질의로 처리하지만, 회사가 운영하지 않는 그래프 DB를 새로 들여야 하고, 근처 관계는 한 단계라 SQL JOIN으로 충분하다.
        - pgvector: 키워드 검색을 뺀 네 질의를 SQL 한 질의로 처리하고, 키워드 검색도 형태소 분석만 앱에서 하면 같은 DB에서 처리한다. 100만 건은 AWS 가이드가 pgvector에 맞는 규모로 보는 수백만 건(장비에 따라) 안이다.
    - 근거 2: 저장 형식(4-3, 검색 질문 45문항):

        | 형식 | 100만 건 벡터(GB) | 맞힌 문항 | Recall@5 | 판정 |
        |---|---|---|---|---|
        | **float32** | **4.100** | **35** | **0.739** | **채택** |
        | float16 | 2.052 | 35 | 0.739 | 메모리가 모자랄 때 첫 대안 |
        | binary | 0.136 | 33 | 0.679 | 낮춘 값만으로는 2문항 잃음 |

        - 100만 건 벡터에 HNSW 인덱스를 더해도(4-2의 3.0배) 약 12.3GB라, 키워드 색인을 더해도 검색용 메모리 64GB보다 한참 작으므로 float32를 유지한다.
- **결과**
    - 얻는 것: 새 DB를 들이지 않고 이미 운영하는 PostgreSQL에 테이블만 더해 의미 검색·키워드 검색·필터·근처 관계·개수 세기를 처리한다. binary(낮춘 값만)보다 2문항 더 맞힌다(33 -> 35문항).
    - 감수하는 것:
        - 한국어 형태소 분석을 앱에서 해 전문 검색 색인에 넣어야 한다.
        - 100만 건에서는 HNSW를 쓰므로 필터 질의에 `hnsw.iterative_scan`을 켜야 한다(4-1).
        - 맞힌 문항이 같은 float16보다 벡터 저장 공간을 두 배 쓴다(100만 건 4.100GB 대 2.052GB).
    - 다시 검토할 조건:
        - 키워드 검색을 PostgreSQL 전문 검색으로 옮김 -> 그 키워드 검색으로 `hybrid`의 Recall@5를 다시 측정한다(3절은 파이썬 `BM25Retriever`로 측정했다).
        - PostgreSQL 전문 검색의 순위가 BM25보다 낮게 측정됨 -> 형태소 분석기를 갖춘 검색 엔진(OpenSearch 등)으로 키워드 검색을 옮긴다.
        - iterative_scan을 켜도 필터 질문의 결과가 모자람 -> 조건 값별 부분 인덱스나 조건 안에서 먼저 찾는 저장소를 다시 비교한다.
        - 관계를 여러 단계 따라가거나 경로 자체를 묻는 질문이 늘어남 -> Neo4j를 후보로 다시 비교한다.
        - 벡터와 인덱스가 서버 한 대의 메모리를 넘음(1억 건이면 벡터만 400GB 이상) -> 전용 벡터 DB를 다시 비교한다.
        - 벡터와 인덱스가 검색용 메모리 64GB를 넘음 -> 이 평가셋에서 잃은 문항이 없었던 float16(`halfvec`)으로 바꾼다.
        - float16으로도 모자람 -> binary(`bit`)로 후보를 뽑고 원본으로 다시 점수를 매기는 방식의 Recall@5를 측정한 뒤 바꾼다.

In [ ]:
connection.close()  # 4절에서만 쓴 Supabase 연결 닫기

## 5. 비용을 셋으로 나누기

3절에서 정확도를 측정했으니 6절 종합 선정에 쓸 비용 축을 계산합니다. 비용을 한 숫자로 합치면 "문서는 많고 질문은 적은 서비스"와 그 반대를 구분할 수 없어 세 가지로 나눕니다.

1. **색인 1회**: 서비스를 열 때 관광지 100만 건을 임베딩하고 BM25 색인을 만듭니다.
2. **월 질의**: 매달 질문 1만 건을 검색합니다.
3. **재색인**: 임베딩 모델을 바꾸면 문서 전체를 다시 임베딩합니다(모델이 다르면 벡터를 섞어 쓸 수 없음). 금액은 새 모델의 색인 1회와 같습니다.

**`monthly_cost` 계산 방법**
- KURE-v1과 BM25는 모두 회사 장비에서 돌기 때문에, 걸린 시간에 장비 1시간 가격(가정)을 곱합니다.
- 색인 1회: 388건 색인에 걸린 시간(2-3)을 문서 수만큼 늘립니다.
- 월 질의: 질문 1건 검색 시간(3-2)에 월 질문 수를 곱합니다.

In [ ]:
machine_usd_per_hour = 0.70  # (가정) 로컬 모델을 돌릴 범용 클라우드 장비 1시간 가격(회사 사정에 맞게 바꿈)

# (가정) 서비스 규모: 관광지 100만 건, 월 질문 1만 건
n_docs = 1_000_000
monthly_queries = 10_000


def monthly_cost(index_seconds_per_doc, query_ms, n_docs, monthly_queries):
    """색인 1회(index_usd)와 월 질의(monthly_query_usd) 비용(달러)을 dict로 반환합니다."""
    index_cost = index_seconds_per_doc * n_docs / 3600 * machine_usd_per_hour
    query_cost = query_ms / 1000 * monthly_queries / 3600 * machine_usd_per_hour
    return {"index_usd": round(float(index_cost), 3), "monthly_query_usd": round(float(query_cost), 3)}


# 의미 검색(KURE-v1)과 BM25 각각의 색인 1회·월 질의 비용
strategy_ms = results.groupby("strategy")["ms"].mean()  # 전략별 질문 1건 평균 검색 시간(ms)
print("KURE-v1 dense:", monthly_cost(index_seconds["KURE-v1"] / len(documents), strategy_ms["dense"], n_docs, monthly_queries))
print("BM25:", monthly_cost(index_seconds["BM25"] / len(documents), strategy_ms["bm25"], n_docs, monthly_queries))

**결과 읽기**
- 두 검색기 모두 색인 1회 비용이 월 질의 비용보다 훨씬 큽니다. 문서가 질문보다 길고 건수도 100배이기 때문입니다.
- 회사 장비에서 도는 로컬 모델이라 금액 자체는 작고, 장비 가격 가정에 따라 달라집니다.

**월 운영비란?**
- 종합 선정의 비용 축으로, 색인 1회 ÷ 12개월 + 월 질의입니다.
- 1년에 한 번 다시 색인한다고 보고, 한 번 내는 돈을 매달 내는 돈과 같은 단위로 맞춘 것입니다.

## 6. 네 축 종합 선정

정확도(3절)와 비용(5절)에 지연·저장 공간까지 모였으니, 이 네 축으로 검색 전략 후보를 비교해 최종 아키텍처 하나를 고릅니다. 순서는 **제약으로 거르기 → 지배당한 후보 지우기 → 고정 기준 가중 점수 → 민감도 확인**이고, 마지막에 결정을 ADR로 남깁니다.

**가정에서 나온 제약**
- 지연 300ms 이하: 검색 단계의 시간 예산
- 저장 64GB 이하: 검색에 쓰는 메모리 합계(pgvector의 벡터·HNSW 인덱스와 BM25 색인)

이 제약을 적용한 경우를 **기본 시나리오**로 삼고, 문서 수 가정을 바꿨을 때 1위가 어떻게 달라지는지는 6-5에서 봅니다.

<img src="images/07_selection_funnel.png" width="1050" alt="후보에서 최종 아키텍처까지 고르는 다섯 단계: 1 제약으로 거르기, 2 지배당한 후보 지우기, 3 가중 점수로 순위 매기기, 4 민감도 확인, 5 ADR에 결정과 다시 검토할 조건 남기기">

### 6-1. 후보표 만들기

후보끼리 비교하려면 네 축 값을 한 테이블에 놓아야 합니다. 후보는 3절의 세 검색 전략이고, 모두 KURE-v1(ADR-001)을 씁니다.

| 후보 | 검색 전략 | 쓰는 검색기 |
|---|---|---|
| A | `dense` | 벡터 저장소 |
| B | `hybrid` | 벡터 저장소 + BM25 색인 |
| C | `bm25` | BM25 색인 |

**네 축 계산 방법**
- 정확도: 3-3의 전체 Recall@5(묘사·이름·조건 45문항)
- 지연(`latency_ms`): 3-2에서 측정한 질문 1건 평균 검색 시간 + 100만 건 검색 시간 30ms(가정). 388건에서 측정한 시간은 100만 건을 대표하지 못해 가정을 더합니다.
- 비용(`cost_usd`): 쓰는 검색기의 색인 시간과 검색 시간으로 구한 월 운영비(5절)
- 저장 공간(`storage_gb`): 검색에 쓰는 메모리만 셉니다. 벡터 값(벡터 수 × 1024차원 × 4바이트)에 4-2에서 측정한 배수(`vector_index_factor`)를 곱한 값과, BM25 색인(문서당 고유 낱말 수 × 항목당 바이트)을 쓰는 검색기만큼 더합니다.

문서 수를 바꾼 시나리오(6-5)에서 다시 쓰도록 후보표를 함수로 만듭니다.

In [ ]:
assumed_search_ms = 30.0  # (가정) 100만 건에서 검색 한 번 시간(ms), 같은 리전의 pgvector 또는 전문 검색 엔진 기준
bm25_bytes_per_term = 12  # (가정) BM25 색인에서 문서 하나의 낱말 하나가 차지하는 바이트
reindex_months = 12  # (가정) 1년에 한 번 재색인: 색인 1회 비용을 이 개월 수로 나눠 월 비용에 넣음
unique_terms_per_doc = float(np.mean([len(set(kiwi_tokenize(doc.page_content))) for doc in documents]))  # 문서당 고유 낱말 수(실측)


def build_candidates(n_docs):
    """문서 수가 n_docs일 때 후보 A~C의 네 축을 담은 후보표를 반환합니다."""
    rows = []
    for name, strategy in [("A", "dense"), ("B", "hybrid"), ("C", "bm25")]:
        uses_vectors = strategy != "bm25"  # 벡터 저장소를 쓰는가
        uses_bm25 = strategy != "dense"  # BM25 색인을 쓰는가
        query_ms = strategy_ms[strategy]  # 3-2에서 측정한 질문 1건 평균 검색 시간

        # 쓰는 검색기만큼 색인 시간과 저장 공간을 더함
        index_seconds_per_doc = (uses_vectors * index_seconds["KURE-v1"] + uses_bm25 * index_seconds["BM25"]) / len(documents)
        storage_gb = (uses_vectors * n_docs * 1024 * 4 * vector_index_factor + uses_bm25 * n_docs * unique_terms_per_doc * bm25_bytes_per_term) / 1e9
        cost = monthly_cost(index_seconds_per_doc, query_ms, n_docs, monthly_queries)

        rows.append({"candidate": name, "strategy": strategy, "accuracy": overall[strategy],
                     "latency_ms": round(query_ms + assumed_search_ms, 3),
                     "cost_usd": round(cost["index_usd"] / reindex_months + cost["monthly_query_usd"], 3),
                     "storage_gb": round(storage_gb, 3)})
    return pd.DataFrame(rows)


candidates = build_candidates(n_docs)
display(candidates)

**결과 읽기**
- 정확도는 B(`hybrid`) 0.794가 가장 높고 A(`dense`) 0.739, C(`bm25`) 0.729 순입니다.
- 지연은 가정한 30ms에 측정한 검색 시간을 더한 값으로, 세 후보 모두 300ms보다 한참 작습니다. A와 B는 몇 ms 차이라 실행할 때마다 순서가 바뀔 수 있습니다(`hybrid`는 `dense` 검색을 포함하므로 실제로는 같거나 깁니다).
- 저장 공간은 벡터를 쓰는 A(12.288GB)·B(13.127GB)가 BM25만 쓰는 C(0.839GB)의 약 15배입니다.

### 6-2. 제약으로 거르기

점수를 매기기 전에 서비스가 받아들일 수 없는 후보부터 빼야, 쓸 수 없는 후보가 점수로 1위가 되는 일을 막습니다.

**제약이란?**
- 점수와 상관없이 지켜야 하는 조건입니다. 기본 시나리오의 두 한도를 `base_limits`에 담고, 값이 한도를 넘는 후보를 뺍니다.

In [ ]:
def apply_constraints(candidates, limits):
    """한도를 넘지 않은 후보 표와, 넘은 후보에 이유(drop_reason)를 붙인 표 두 개를 반환합니다."""
    passed_rows = []
    dropped_rows = []
    for row in candidates.to_dict("records"):
        reasons = [f"{column}>{limit}" for column, limit in limits.items() if row[column] > limit]  # 한도는 상한
        if reasons:
            row["drop_reason"] = ", ".join(reasons)
            dropped_rows.append(row)
        else:
            passed_rows.append(row)
    return pd.DataFrame(passed_rows), pd.DataFrame(dropped_rows, columns=list(candidates.columns) + ["drop_reason"])  # 탈락 후보가 없어도 candidate 열이 있도록


base_limits = {"latency_ms": 300, "storage_gb": 64}  # 기본 시나리오 제약: 지연 300ms, 저장 64GB

passed, dropped = apply_constraints(candidates, base_limits)
print("통과:", list(passed["candidate"]), "| 탈락:", list(dropped["candidate"]))

### 6-3. 지배당한 후보 지우기

제약을 통과한 후보 가운데에도 가중치를 어떻게 정해도 1위가 될 수 없는 후보가 있습니다. 이런 후보를 먼저 지우면 6-4의 점수 비교가 실제로 경쟁하는 후보로 좁혀집니다.

**지배당한 후보란?**
- 후보 X가 Y보다 **모든 축에서 같거나 나쁘고 한 축 이상에서 나쁘면** X는 Y에게 **지배당합니다**. 가중치를 어떻게 줘도 Y보다 높은 점수를 받을 수 없습니다.
- 지배당한 후보를 지우고 남은 후보를 **파레토 후보**라고 합니다.
- `axes`에 축마다 좋은 방향(`max`, `min`)을 적습니다.

In [ ]:
axes = {"accuracy": "max", "latency_ms": "min", "cost_usd": "min", "storage_gb": "min"}  # 네 축과 좋은 방향


def remove_dominated(candidates, axes):
    """지배당한 후보를 지운 표와 {지배당한 후보: [이긴 후보 목록]} dict를 반환합니다."""
    rows = candidates.to_dict("records")
    kept = []
    dominated = {}
    for row in rows:
        winners = []
        for other in rows:
            if other["candidate"] == row["candidate"]:
                continue
            # 축마다 other가 얼마나 좋은지(작을수록 좋은 축은 부호를 뒤집음)
            differences = [(other[axis] - row[axis]) * (1 if direction == "max" else -1) for axis, direction in axes.items()]
            # 모든 축에서 같거나 좋고, 한 축 이상에서 좋으면 other가 row를 지배
            if min(differences) >= 0 and max(differences) > 0:
                winners.append(other["candidate"])
        if winners:
            dominated[row["candidate"]] = winners
        else:
            kept.append(row)
    return pd.DataFrame(kept), dominated


front, dominated = remove_dominated(passed, axes)
print("파레토 후보:", list(front["candidate"]), "| 지배당함(이긴 후보):", dominated)

**결과 읽기**
- 세 후보 모두 파레토 후보입니다. B는 정확도가, C는 지연·비용·저장 공간이 가장 좋고, A는 B보다 비용과 저장 공간이 작고 C보다 정확도가 높아(0.739 대 0.729) 어느 후보에게도 지배당하지 않습니다.

### 6-4. 고정 기준으로 점수 매기기

남은 파레토 후보는 축마다 더 좋은 후보가 달라서, 네 축을 한 점수로 합쳐야 1위를 정할 수 있습니다.

**고정 기준 점수란?**
- 작을수록 좋은 축은 미리 정한 **고정 기준**(받아들일 수 있는 최대치)으로 `1 - 값 / 기준`으로 바꿉니다(기준을 넘으면 0점).
- 정확도는 이미 0~1입니다.

**min-max 정규화는 왜 쓰지 않나?**
- min-max 정규화(테이블의 최솟값 1점, 최댓값 0점)는 후보 하나를 더하거나 빼면 다른 후보들의 점수와 순위가 바뀝니다.
- 예를 들어 지연 30ms와 37ms 두 후보만 남으면 7ms 차이가 1점 대 0점이 됩니다.

**여행지 검색의 기준과 가중치(팀 합의)**
- 지연 300ms: 답변 생성 전 검색 단계의 시간 예산(제약과 같은 값)
- 월 비용 $1,000: 검색 쪽 월 예산
- 저장 64GB: 검색에 쓰는 메모리 합계(제약과 같은 값)
- 가중치: 정확도 0.7, 나머지 각 0.1

**환율로 검산하기**
- 정한 뒤에는 **환율**(정확도 1%p와 같은 점수가 되는 각 축의 양)로 검산합니다.
- 정확도 1%p는 점수를 `0.01 × 정확도 가중치`만큼 올리고 축 값 1은 점수를 `축 가중치 ÷ 기준`만큼 깎으므로, 앞을 뒤로 나눈 값이 같은 점수가 되는 축의 양입니다.

In [ ]:
references = {"latency_ms": 300.0, "cost_usd": 1000.0, "storage_gb": 64.0}  # 작을수록 좋은 축의 고정 기준
weights = {"accuracy": 0.7, "latency_ms": 0.1, "cost_usd": 0.1, "storage_gb": 0.1}  # 축별 가중치(합 1)


def score_candidates(candidates, weights, references):
    """축마다 0~1 점수로 바꾸고 가중치를 곱해 더한 점수표(점수 높은 순)를 반환합니다."""
    rows = []
    for row in candidates.to_dict("records"):
        scored = {"candidate": row["candidate"]}
        total = 0.0
        for axis, weight in weights.items():
            # 작을수록 좋은 축은 1 - 값/기준(기준을 넘으면 0점), 정확도는 그대로
            value = max(0.0, 1 - row[axis] / references[axis]) if axis in references else row[axis]
            scored[axis + " 점수"] = round(value, 3)
            total += weight * value
        scored["score"] = total  # 격차를 정확히 계산하려고 반올림하지 않음(화면에서만 반올림)
        rows.append(scored)
    return pd.DataFrame(rows).sort_values(["score", "candidate"], ascending=[False, True]).reset_index(drop=True)  # 점수 높은 순, 동점이면 이름 순


# 환율: 정확도 1%p와 같은 점수가 되는 지연 ms, 월 비용 $, 저장 GB
exchange = {axis: round(0.01 * weights["accuracy"] / (weights[axis] / references[axis]), 3) for axis in references}
print("정확도 1%p와 같은 점수:", exchange)

scored = score_candidates(front, weights, references)
display(scored.round(3))

**결과 읽기**
- 정확도 1%p는 지연 21ms, 월 비용 $70, 저장 4.48GB와 같습니다. 팀이 이 환율에 동의하지 않으면 가중치나 기준을 바꿉니다.
- B가 0.820으로 1위, C가 0.799, A가 0.783입니다. C는 지연·저장 점수에서 앞서지만 정확도 가중치가 0.7이라 정확도가 0.065 높은 B가 앞섭니다(1위와 2위의 정확한 점수 차는 6-5 표의 `격차`에 나옵니다).
- 비용 점수는 모든 후보가 1.0입니다. 월 비용이 기준 $1,000에 비해 아주 작아 순위에 영향을 주지 않습니다.

### 6-5. 가정이 바뀌면 1위가 바뀌는가

6-4의 1위는 기본 시나리오의 가정 위에서 나온 결과라, 어느 가정이 1위를 바꾸는지 알아야 ADR 결과 칸에 다시 검토할 조건을 적을 수 있습니다. 세 단계를 `run_selection`으로 묶어 시나리오마다 다시 실행하고, 1위와 2위의 점수 차(`격차`)를 봅니다.

**시나리오**
- 기본: 문서 100만 건
- 문서 1,000만 건: 같은 서비스가 열 배로 커질 때

In [ ]:
def run_selection(candidates, limits, weights, references):
    """제약 -> 지배 제거 -> 고정 기준 점수를 실행해 탈락·지배·점수표와 1위·격차를 dict로 반환합니다."""
    passed, dropped = apply_constraints(candidates, limits)
    front, dominated = remove_dominated(passed, axes)
    scored = score_candidates(front, weights, references)
    gap = round(float(scored.loc[0, "score"] - scored.loc[1, "score"]), 4) if len(scored) > 1 else None  # 1위와 2위의 점수 차
    return {"dropped": dropped, "dominated": dominated, "scored": scored, "winner": scored.loc[0, "candidate"], "gap": gap}


# 시나리오마다 후보표를 만들어 선정을 다시 실행
scenario_rows = []
for label, scenario_docs in [("기본: 문서 100만 건", 1_000_000), ("문서 1,000만 건", 10_000_000)]:
    result = run_selection(build_candidates(scenario_docs), base_limits, weights, references)
    scenario_rows.append({"시나리오": label, "제약 탈락": list(result["dropped"]["candidate"]), "지배당함": list(result["dominated"]),
                          "파레토 후보": list(result["scored"]["candidate"]), "1위": result["winner"], "격차": result["gap"]})
display(pd.DataFrame(scenario_rows))

**결과 읽기**
- 기본 시나리오(문서 100만 건)의 1위는 B이고 격차는 약 0.02입니다. 격차의 넷째 자리는 측정한 검색 시간에 따라 실행마다 조금씩 바뀝니다.
- 문서가 1,000만 건이 되면 벡터를 쓰는 A·B의 저장 공간이 64GB를 넘어 제약에서 탈락하고 C만 남습니다. 비교할 2위가 없어 격차는 `NaN`입니다. 1위를 바꾸는 가정은 문서 수입니다.

**민감도**: 시나리오는 가정을 바꿔 봤으니, 팀이 합의한 가중치가 조금 달라져도 결정이 그대로인지 봅니다. 가중치를 하나씩 ±20% 바꾸고(합은 다시 1로) 기본 시나리오의 1위가 유지되는지 확인합니다.

In [ ]:
def check_sensitivity(candidates, limits, weights, references):
    """가중치 하나씩 0.8배·1.2배로 바꾼 조합마다 1위와 격차를 담은 표를 반환합니다."""
    rows = []
    for axis in weights:
        for factor in [0.8, 1.2]:
            changed = dict(weights)
            changed[axis] = weights[axis] * factor
            total = sum(changed.values())
            changed = {name: value / total for name, value in changed.items()}  # 합을 다시 1로
            result = run_selection(candidates, limits, changed, references)
            rows.append({"changed_weight": f"{axis} x{factor}", "winner": result["winner"], "gap": result["gap"]})
    return pd.DataFrame(rows)


sensitivity = check_sensitivity(candidates, base_limits, weights, references)
print("1위 분포:", sensitivity["winner"].value_counts().to_dict(), "| 가장 작은 격차:", sensitivity["gap"].min())

**결과 읽기**
- 가중치를 하나씩 ±20% 바꾼 8가지 조합 모두 B가 1위이고, 가장 작은 격차는 약 0.015입니다. 기본 시나리오의 결정은 가중치에 흔들리지 않습니다.
- 1위가 조합마다 갈리면 **불안정한 결정**이므로, ADR 결과 칸의 다시 검토할 조건에 "어느 가정이 바뀌면 다른 후보로 간다"를 적습니다.

#### ADR-004: 검색 구성으로 KURE-v1 + hybrid + pgvector 사용

- **상태**: 채택
- **맥락**: 임베딩 모델(ADR-001, KURE-v1), 검색 전략(ADR-002, `hybrid`), 벡터 저장소(ADR-003, pgvector)를 각각 정했다. 문서 100만 건, 월 질문 1만 건, 지연 300ms·메모리 64GB 제약 아래에서 정확도·지연·월 비용·저장 공간 네 축으로 검색 전략 후보(`dense`, `hybrid`, `bm25`)를 다시 비교해 최종 구성을 확정해야 한다.
- **결정**: 검색 구성으로 KURE-v1 임베딩, `hybrid` 검색, pgvector(float32) 저장소를 쓰고 ADR-001~003을 유지한다.
    - 근거: 기본 시나리오(문서 100만 건)의 후보표와 고정 기준 점수(정확도 0.7, 지연·비용·저장 각 0.1):

        | 후보 | 정확도 | 저장(GB) | 점수 | 판정 |
        |---|---|---|---|---|
        | A `dense` | 0.739 | 12.288 | 0.783 | 점수 3위 |
        | C `bm25` | 0.729 | 0.839 | 0.799 | 점수 2위 |
        | **B `hybrid`** | **0.794** | **13.127** | **0.820** | **채택** |

        - 세 후보 모두 제약을 통과하고 지배당한 후보가 없다.
        - 월 비용은 세 후보 모두 $1 안이라 점수 차이를 만들지 않는다(비용 점수 모두 1.0). 지연은 C 쪽으로 약 0.005를 더한다.
        - A: B보다 저장 공간이 0.839GB 작지만 정확도가 0.055 낮다.
        - C: 지연·저장 점수는 가장 높지만 정확도가 0.065 낮아 B보다 점수가 약 0.02 낮다.
        - B: 가중치를 하나씩 ±20% 바꾼 8조합 모두 1위다(가장 작은 격차 약 0.015).
- **결과**
    - 얻는 것: C보다 정확도가 0.065(0.729 -> 0.794), A보다 0.055(0.739 -> 0.794) 높다.
    - 감수하는 것: 키워드 색인과 벡터 인덱스를 함께 운영하고, 검색 메모리를 키워드 검색만 쓸 때(0.839GB)보다 많은 13.127GB 쓴다.
    - 다시 검토할 조건:
        - 문서가 약 490만 건을 넘음(B의 저장 공간이 64GB에 닿음, 6-5의 1,000만 건에서는 A·B가 탈락) -> 4-3의 float16·binary로 저장 공간을 줄인 후보를 넣어 다시 선정한다.
        - 팀 합의 가중치가 ±20%보다 크게 바뀜(특히 정확도 가중치를 낮춤) -> 2위 C(`bm25`)와 다시 비교한다.
        - 운영에서 `hybrid`의 검색 시간이 가정보다 크게 늘어남(두 검색을 차례로 실행) -> 지연을 다시 측정해 다시 선정한다.
        - 평가 질문이나 관광지 문서가 바뀜 -> 3절의 유형 × 전략 표부터 다시 측정한다.

**확인 퀴즈**: 후보 하나를 더 넣었더니 기존 두 후보의 순위가 뒤바뀌었습니다. 어떤 정규화를 썼을 가능성이 크고, 어떻게 고칠까요?

<details><summary>정답 보기</summary>

min-max 정규화일 가능성이 큽니다. 새 후보가 어떤 축의 최솟값이나 최댓값을 바꾸면 모든 후보의 그 축 점수가 다시 계산됩니다. 축마다 고정 기준을 정해 `1 - 값 / 기준`으로 바꾸면 한 후보의 점수가 다른 후보와 상관없이 정해집니다.

</details>

## 7. 🖐️ 함께 따라하기: 사내 노무 상담 RAG 종합 선정과 ADR

본실습의 절차(서비스가 보낼 질의로 저장소 정하기, 검색 전략 측정, 제약·지배·점수·민감도로 선정하기, ADR 작성)를 질문과 기준이 다른 노무 상담 봇에 적용해 직접 결정을 내립니다. 본실습 2~6절을 실행한 상태에서 진행합니다(4-2의 `vector_index_factor`와 6절 함수를 그대로 씀).
- 임베딩 모델은 교안_01 따라하기의 노무 ADR-001에서 고른 KURE-v1을 그대로 씁니다.
- 노무 질문 유형은 본실습 유형에 이렇게 대응합니다: 직원 말투는 묘사, 조문번호는 이름, 필터는 조건, 법+매뉴얼·2단계 관계는 관계(집계 유형은 없음).
- 노무 질문 다섯 유형 가운데 검색으로 답하는 직원 말투·조문번호·필터는 세 검색 전략(`dense`·`hybrid`·`bm25`)으로 측정해 고릅니다.
- 법+매뉴얼·2단계 관계 질문은 조문 사이 관계를 따라가는 구조화 질의가 필요해 측정하지 않고, 그 질의를 받을 수 있는 저장소인지만 봅니다(본실습 1-1과 같은 기준).

**상황**: 인사팀 상담 봇은 근로기준법·시행령 조문과 유연근무 매뉴얼에서 근거를 찾아 답합니다. 회사는 이미 PostgreSQL을 운영합니다.

**선정 기준(팀 합의)**
- 한도: 검색 단계 지연 300ms, 검색 서버 메모리 4GB
- 고정 기준: 지연 300ms, 검색 쪽 월 예산 $50, 메모리 4GB
- 가중치는 직접 정합니다.

진행 순서:

1. 7-1 질문 유형으로 저장소 정하고 노무 ADR-002 남기기(부분 작성)
2. 7-2 노무 검색 전략 후보표와 가중치·한도, 노무 ADR-003 남기기(부분 작성)

**실습 진행 방법**

작성 셀은 이 함께 따라하기 절에만 있습니다.

- **부분 작성:** 코드의 `...` 자리나 마크다운의 괄호 안 안내만 채웁니다. 이미 적힌 코드는 그대로 실행합니다.
- `...`가 남은 채로 실행하면 에러가 나거나 빈 결과가 나옵니다. 작성 셀을 완성해 실행하면 결과가 그 셀에 함께 나옵니다.
- 완성 코드와 답은 수업이 끝난 뒤 강사가 공유합니다. 먼저 직접 작성해 봅니다.

### 7-1. 질문 유형으로 저장소 정하고 노무 ADR-002 남기기

본실습 4-1처럼 저장소는 서비스가 보낼 질의로 고르므로, 노무 질문이 유형별로 몇 문항인지부터 셉니다. 노무는 조문번호 질문이 있어 키워드 검색(BM25)이 필요하다고 보고 저장소를 먼저 정한 뒤, 7-2에서 검색 전략을 측정해 고릅니다. 노무 문서는 7-2에서 검색기를 만들 때도 씁니다.

In [ ]:
# 노무 문서(JSON Lines)와 평가 질문 읽기
labor_docs = [json.loads(line) for line in (data_dir / "따라하기" / "docs.jsonl").read_text(encoding="utf-8").splitlines()]
labor_questions = json.loads((data_dir / "따라하기" / "questions.json").read_text(encoding="utf-8"))
print("노무 문서:", len(labor_docs), "| 질문:", len(labor_questions))

display(pd.DataFrame(labor_questions)["question_type"].value_counts().rename("질문 수").to_frame())  # 유형별 질문 수

**부분 작성 · 노무 질의표와 저장소**

**작성:** 아래 마크다운 셀을 더블클릭해 `( )` 칸과 괄호 안 안내를 채웁니다.

1. Chroma, pgvector, Neo4j Vector 칸: 본실습 4-1처럼 "한 질의 / 코드로 우회 / 불가"로 쓰고 괄호에 방법을 짧게 적습니다.
   - 조문 관계는 `article_refs.csv`(문서 사이 `참조`·`준용`·`근거` 관계)를 pgvector에서는 관계 테이블, Neo4j에서는 `REFERS` 관계로 넣는다고 봅니다.
   - 관계 두 단계를 SQL로 따라가려면 JOIN을 두 번 하거나, 단계 수를 정해 두지 않으려면 앞 단계 결과에서 다시 한 단계 나가기를 되풀이하는 **재귀 CTE**(`WITH RECURSIVE`)를 써야 합니다. Cypher는 가변 길이 경로 하나로 끝납니다.
2. 저장 형식: 노무 문서 1만 건의 KURE-v1 벡터(1만 × 1024차원 × 4바이트, HNSW 인덱스를 더하면 본실습 4-2의 배수만큼)가 검색 서버 메모리 4GB에 들어가는지 계산해, float32를 유지할지 본실습 4-3처럼 낮출지 정합니다.
3. 노무 ADR-002: 위 유형별 질문 수와 상황(회사 PostgreSQL)을 보고 저장소와 저장 형식을 정해 다섯 칸을 채웁니다.

**반환값:** 다섯 유형의 세 칸이 채워진 표와 다섯 칸이 채워진 노무 ADR-002입니다.

| 유형(필요한 처리) | 예시 질문 | 필요한 질의 | Chroma | pgvector | Neo4j Vector |
|---|---|---|---|---|---|
| 직원 말투(의미 검색) | 하루 네 시간이나 여덟 시간 일하면 중간에 얼마나 쉬어야 하나요? | 질문과 가까운 벡터 k개 | ( ) | ( ) | ( ) |
| 조문번호(키워드 검색) | 근로기준법 제53조 연장근로 제한·주 12시간 한도·특별연장근로 요건 | 조문 번호 키워드 검색 | ( ) | ( ) | ( ) |
| 필터(Self-Query가 뽑은 필터) | 통상임금의 의미와 ... 근로기준법 시행령에서는 어떻게 규정하고 있나요? | 문서 종류(`doc_type`, `topic`) 조건 안에서 벡터 k개 | ( ) | ( ) | ( ) |
| 법+매뉴얼(관계 한 단계) | 시차출퇴근을 도입할 때 취업규칙에는 어떤 운영사항을 정해 두는 것이 실무상 좋으며, ... 고용노동부 신고 의무는 어떻게 되나요? | 매뉴얼에서 근거 조문으로 관계 한 단계 | ( ) | ( ) | ( ) |
| 2단계 관계(관계 두 단계) | 근로기준법 시행령 제17조에서 정한 지연이자 이율은 ... 그 기한을 늘릴 수 있는 경우도 알려 주세요. | 조문 관계 두 단계 | ( ) | ( ) | ( ) |

#### 노무 ADR-002: (고른 기술이 드러나는 제목)

- **상태**: (제안·채택·거절·대체됨 가운데 하나)
- **맥락**: (2~3문장: 정할 문제, 질문 유형과 관계 깊이, 기존 인프라, 검토한 후보. 결정은 쓰지 않음)
- **결정**: (고른 저장소와 저장 형식을 한 문장으로: …를 쓰고, 벡터는 …로 저장한다)
    - 근거: (후보마다 한 줄: 한 질의로 처리하는지와 뺀 이유. 마지막 줄은 고른 저장소가 이긴 이유)
- **결과**
    - 얻는 것: (이 결정으로 얻는 것)
    - 감수하는 것: (대신 감수하는 것)
    - 다시 검토할 조건: (바뀌는 가정이나 수치 -> 다시 할 일)

### 7-2. 노무 검색 전략 후보표와 가중치·한도, 노무 ADR-003 남기기

저장소를 정했으니 검색 전략을 고릅니다. 본실습 2-3·6-1과 같은 방법으로 노무 문서의 검색기를 만들고, 세 전략을 검색으로 답하는 노무 질문 18개(직원 말투·조문번호·필터)로 측정해 후보표를 만듭니다.
- 노무 문서도 조문·매뉴얼 항목 하나를 통째로 임베딩해 문서 수와 벡터 수가 같습니다.
- 규모는 문서 1만 건, 월 질문 5천 건으로 가정합니다.

In [ ]:
# 노무 문서마다 Document 하나(원문 ID를 source_id에)
labor_documents = [Document(id=doc["doc_id"], page_content=doc["title"] + "\n" + doc["text"], metadata={"source_id": doc["doc_id"]})
                   for doc in labor_docs]

labor_index_seconds = {}  # 검색기별 색인 시간(초)

# BM25 색인
start = time.perf_counter()
labor_bm25 = BM25Retriever.from_documents(labor_documents, preprocess_func=kiwi_tokenize, k=10)
labor_index_seconds["BM25"] = time.perf_counter() - start

# KURE-v1 벡터를 Chroma 메모리 컬렉션에 넣기
labor_store = Chroma(collection_name="labor_docs", embedding_function=embedding_model)
labor_store.reset_collection()
start = time.perf_counter()
labor_store.add_documents(labor_documents)
labor_index_seconds["KURE-v1"] = time.perf_counter() - start

# 세 검색기(본실습 3-1과 같은 설정)
labor_dense = labor_store.as_retriever(search_kwargs={"k": 10})
labor_retrievers = {"dense": labor_dense, "bm25": labor_bm25,
                    "hybrid": EnsembleRetriever(retrievers=[labor_bm25, labor_dense], weights=[0.5, 0.5], c=60, id_key="source_id")}
print("노무 문서:", len(labor_documents), "| BM25 색인(초):", round(labor_index_seconds["BM25"], 3), "| KURE-v1 임베딩과 적재(초):", round(labor_index_seconds["KURE-v1"], 3))

In [ ]:
# (가정) 사내 문서 1만 건, 월 질문 5천 건
labor_n_docs = 10_000
labor_monthly_queries = 5_000
labor_unique_terms = float(np.mean([len(set(kiwi_tokenize(doc.page_content))) for doc in labor_documents]))  # 문서당 고유 낱말 수

# 측정에 쓸 노무 질문: 검색으로 답하는 유형만(관계 두 유형은 쓰지 않음)
labor_search_types = ["직원 말투", "조문번호", "필터"]
labor_search_questions = [question for question in labor_questions if question["question_type"] in labor_search_types]

labor_dense.invoke(labor_search_questions[0]["question"])  # 첫 검색의 준비 시간이 측정에 섞이지 않게 한 번 먼저 실행

# 전략마다 노무 질문 18개를 검색해 네 축을 계산(본실습 6-1과 같은 식)
labor_rows = []
for name, mode in [("A", "dense"), ("B", "hybrid"), ("C", "bm25")]:
    recalls = []
    times = []
    for question in labor_search_questions:
        start = time.perf_counter()
        found = labor_retrievers[mode].invoke(question["question"])
        times.append((time.perf_counter() - start) * 1000)
        recalls.append(recall_at_k([doc.metadata["source_id"] for doc in found], question["relevant_doc_ids"], 5))

    uses_vectors = mode != "bm25"  # 벡터 저장소를 쓰는가
    uses_bm25 = mode != "dense"  # BM25 색인을 쓰는가
    index_seconds_per_doc = (uses_vectors * labor_index_seconds["KURE-v1"] + uses_bm25 * labor_index_seconds["BM25"]) / len(labor_documents)
    cost = monthly_cost(index_seconds_per_doc, float(np.mean(times)), labor_n_docs, labor_monthly_queries)
    storage_gb = (uses_vectors * labor_n_docs * 1024 * 4 * vector_index_factor + uses_bm25 * labor_n_docs * labor_unique_terms * bm25_bytes_per_term) / 1e9
    labor_rows.append({"candidate": name, "strategy": mode, "accuracy": round(float(np.mean(recalls)), 3),
                       "latency_ms": round(float(np.mean(times)) + assumed_search_ms, 3),
                       "cost_usd": round(cost["index_usd"] / reindex_months + cost["monthly_query_usd"], 3),
                       "storage_gb": round(storage_gb, 3)})
labor_candidates = pd.DataFrame(labor_rows)
display(labor_candidates)

노무 상담 봇의 1위는 이 업무가 무엇을 중시하는지(가중치)와 무엇을 넘으면 안 되는지(한도)로 정해집니다. 고정 기준은 상황의 팀 합의값을 그대로 쓰고, 가중치와 한도를 정해 환율로 검산한 뒤 선정을 실행합니다.

**부분 작성 · 가중치와 한도**

**작성:**

1. `labor_weights`: 본실습 `weights`와 같은 네 키(합 1)로 노무 상담 봇의 가중치를 정하고, 주석에 이유를 한 줄 씁니다.
2. `labor_limits`: 본실습 `base_limits`와 같은 모양의 dict로 팀 합의 한도(검색 단계 300ms, 검색 서버 메모리 4GB)를 담습니다.

**반환값:** 환율 한 줄, 1위·격차·지배당한 후보 한 줄, 점수표, `민감도 1위 분포`가 출력됩니다.

In [ ]:
# 고정 기준은 팀 합의값(검색 단계 300ms, 검색 쪽 월 예산 $50, 검색 서버 메모리 4GB)
labor_references = {"latency_ms": 300.0, "cost_usd": 50.0, "storage_gb": 4.0}

# 1. 가중치(합 1)를 정하고 이유를 한 줄 쓰세요.
# 이유: ...
labor_weights = ...  # {"accuracy": ..., "latency_ms": ..., "cost_usd": ..., "storage_gb": ...}

# 2. 팀 합의 한도를 dict로 쓰세요(본실습 base_limits와 같은 모양).
labor_limits = ...

# 환율: 정확도 1%p와 같은 점수가 되는 각 축의 양(본실습과 같은 식)
labor_exchange = {axis: round(0.01 * labor_weights["accuracy"] / (labor_weights[axis] / labor_references[axis]), 3) for axis in labor_references}
print("정확도 1%p와 같은 점수:", labor_exchange)

# 한도 -> 지배 제거 -> 고정 기준 점수
labor_choice = run_selection(labor_candidates, labor_limits, labor_weights, labor_references)
print("1위:", labor_choice["winner"], "| 격차:", labor_choice["gap"], "| 지배당함:", labor_choice["dominated"])
display(labor_choice["scored"].round(3))

# 민감도: 가중치를 하나씩 0.8배·1.2배로 바꿔도 1위가 유지되는지 보기
print("민감도 1위 분포:", check_sensitivity(labor_candidates, labor_limits, labor_weights, labor_references)["winner"].value_counts().to_dict())

- 후보표에서 정확도가 가장 높은 후보와 지연·저장 공간이 가장 작은 후보가 다르면, 가중치가 둘 사이를 정합니다.
- `민감도 1위 분포`에 후보가 하나뿐이면 가중치를 조금 바꿔도 결정이 그대로입니다.

**부분 작성 · 노무 ADR-003**

노무 ADR-001(임베딩 모델)과 노무 ADR-002(벡터 저장소) 위에서 고른 검색 전략까지 묶어 최종 아키텍처를 남깁니다.

**작성:** 아래 마크다운 셀을 더블클릭해 괄호 안 안내를 지우고 다섯 칸을 채웁니다.

1. 상태: 제안·채택·거절·대체됨 가운데 하나
2. 맥락: 노무 ADR-001·002 위에 더한 규모와 팀 합의 고정 기준
3. 결정: 임베딩 모델, 검색 전략, 저장소(앞 ADR을 유지하는지)와 그 근거(위 출력의 1위와 격차, 가중치를 바꿔도 같은지)
4. 결과: 얻는 것과 감수하는 것, 그리고 다시 검토할 조건을 "무엇이 바뀌면 무엇을 다시 한다" 꼴로 3개 이상(예: 근로기준법 개정 시행, 관계 질문 처리 방식이 정해질 때)

**반환값:** 다섯 칸이 모두 채워진 노무 ADR-003입니다.

#### 노무 ADR-003: (고른 기술이 드러나는 제목)

- **상태**: (제안·채택·거절·대체됨 가운데 하나)
- **맥락**: (2~3문장: 정할 문제, 규모와 팀 합의 고정 기준, 검토한 후보. 결정은 쓰지 않음)
- **결정**: (임베딩 모델, 검색 전략, 저장소를 한 문장으로: …로 확정하고 앞 ADR을 유지하는지)
    - 근거: (후보마다 한 줄: 점수와 진 이유. 마지막 줄은 1위의 격차와 가중치를 바꿔도 같은지)
- **결과**
    - 얻는 것: (이 결정으로 얻는 것)
    - 감수하는 것: (대신 감수하는 것)
    - 다시 검토할 조건: (바뀌는 가정이나 수치 -> 다시 할 일, 3개 이상)

같은 방법이라도 질문 분포와 제약이 다르면 여행지 검색과 결론이 다를 수 있습니다.

**확인 퀴즈**: 근로기준법 시행령만 개정되어 일부 조문의 내용과 시행일이 바뀌었습니다. 무엇을 다시 해야 할까요?

<details><summary>정답 보기</summary>

바뀐 조문만 다시 임베딩합니다(모델을 바꾼 것이 아니므로 전체 재임베딩은 필요 없습니다). BM25를 함께 쓰는 구성이면 BM25 색인은 전체를 다시 만듭니다. 낱말이 얼마나 드문지를 전체 문서로 계산하기 때문입니다. `effective_date`(문서 metadata의 시행일)와 `article_refs`도 고친 뒤 평가셋을 다시 돌리고, 결과가 달라지면 선정을 다시 해 새 ADR을 남깁니다.

</details>

## 8. 핵심 정리

검색 전략부터 최종 아키텍처까지 내린 결정을 다른 서비스에서도 같은 순서로 되풀이할 수 있게 요점을 모읍니다.

- 질문 유형으로 처리 방식을 나눕니다. 문서를 찾으면 답이 나오는 질문(묘사·이름·조건)은 검색 전략으로, 관계를 따라가거나 개수를 세는 질문은 구조화 질의(Text-to-SQL·Text2Cypher)로 답합니다.
- 검색 전략은 유형 × 전략 표로 측정해 고르고, 유형마다 다른 전략을 쓰는 것은 라우터를 둘 만큼 정확도가 오를 때만 고릅니다. 임베딩 모델을 바꾸면 다시 측정합니다.
- 조건 질문의 필터는 Self-Query로 질문에서 뽑아 붙입니다. 저장소는 그 필터를 받을 수 있어야 합니다.
- 벡터 저장소는 서비스가 보낼 질의(벡터 검색, 키워드 검색, 메타데이터 필터, 관계 탐색, 개수 세기)를 한 질의로 처리하는지로 고르고, 둘 이상이면 이미 운영하는 DB를 먼저 봅니다.
- 저장 형식은 정밀도마다 정확도와 저장 공간을 측정해 정합니다. 메모리에 들어가면 float32를 유지하고, 모자라면 손실이 작은 형식부터 낮춥니다.
- pgvector는 HNSW와 `WHERE`를 함께 쓰면 사후 필터라 결과가 요청 수보다 적을 수 있으므로, 필터 질의에는 `hnsw.iterative_scan`을 켜고 결과 수를 셉니다.
- 비용은 색인 1회, 월 질의, 재색인으로 나누고, 선정에는 월 운영비를 씁니다.
- 선정은 제약, 지배 제거, 고정 기준 점수, 민감도 순서이며, 환율로 가중치를 검산합니다.
- 1위를 바꿀 수 있는 가정을 ADR 결과 칸의 다시 검토할 조건에 남깁니다.

**실무에서 고르는 순서**

1. 질문에 유형 라벨과 정답 문서를 붙인 평가셋을 만들고, 검색으로 답하는 유형과 구조화 질의가 필요한 유형을 나눕니다.
2. 검색으로 답하는 유형에서 `bm25`·`dense`·`hybrid`를 측정해 전략을 고릅니다(같으면 단순한 쪽).
3. 조건 필터는 Self-Query, 관계·집계는 Text-to-SQL·Text2Cypher로 처리하고, 실제 서비스 질문으로 질의 분석 정확도를 따로 측정합니다.
4. 서비스가 보낼 질의로 저장소를 고르되 이미 운영하는 DB를 먼저 검토하고, 필터 질문은 결과 수를 세며, 인덱스가 서버 한 대의 메모리를 넘는 규모면 전용 벡터 DB를 검토합니다.
5. 저장 형식은 정밀도별 정확도와 저장 공간을 측정해 정합니다.
6. 후보표를 만들어 제약, 지배, 점수, 민감도로 고르고 ADR을 남깁니다.

고른 아키텍처에 앞 단원 기법을 어디에 붙일지는 부록(RAG 단계별 기법 체계)에 정리했습니다.

다음 시간에는 이렇게 고른 RAG가 만든 답변을 평가하는 방법을 다룹니다.